In [ ]:
#@title 1. Instalar dependências (PyTorch já existe no Colab)
%pip install -q "gymnasium[classic-control]>=1.0" numpy pandas matplotlib


# DQN e Double DQN em Classic Control

Notebook executado no Google Colab para comparar DQN e DDQN em `CartPole-v1` e `MountainCar-v0`. As células registram configurações, resultados por semente, avaliações sem exploração e figuras.


In [ ]:
#@title 2. Importações e pasta de resultados
import json
import math
import random
import shutil
import time
from collections import deque, namedtuple
from copy import deepcopy
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn

PASTA_RESULTADOS = Path("/content/resultados_dqn_ddqn")
if PASTA_RESULTADOS.exists():
    shutil.rmtree(PASTA_RESULTADOS)
PASTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Dispositivo:", DISPOSITIVO)
print("Gymnasium:", gym.__version__, "| PyTorch:", torch.__version__)
print("Resultados em:", PASTA_RESULTADOS)


## Implementação

DQN e DDQN usam a mesma MLP, replay buffer e rede-alvo. No DQN, a rede-alvo seleciona o maior valor do próximo estado; no DDQN, a rede online seleciona a ação e a rede-alvo a avalia.


In [ ]:
#@title 3. CONFIGURAÇÃO QUE VOCÊ ALTERA
# Escolha UM ambiente por execução. MountainCar é opcional e mais demorado.
AMBIENTE = "CartPole-v1"  #@param ["CartPole-v1", "MountainCar-v0"]

# Comece só com baseline. Depois, por exemplo:
# CONFIGURACOES_ESCOLHIDAS = ["baseline", "exploracao_rapida", "learning_rate_alta", "target_rapido", "target_lento"]
CONFIGURACOES_ESCOLHIDAS = ["baseline"]
SEMENTES = [0, 1, 2]
ALGORITMOS = ["DQN", "DDQN"]
EPISODIOS_AVALIACAO = 20

# Perfis-base adequados a cada ambiente.
PERFIS = {
    "CartPole-v1": {
        "episodios": 400,
        "camadas": [128, 128],
        "lr": 1e-3,
        "gamma": 0.99,
        "buffer": 50_000,
        "warmup": 1_000,
        "batch": 64,
        "epsilon_inicio": 1.0,
        "epsilon_fim": 0.05,
        "epsilon_decay_passos": 30_000,
        "target_update": 250,
        "grad_clip": 10.0,
    },
    "MountainCar-v0": {
        "episodios": 1_000,
        "camadas": [128, 128],
        "lr": 1e-3,
        "gamma": 0.99,
        "buffer": 100_000,
        "warmup": 2_000,
        "batch": 64,
        "epsilon_inicio": 1.0,
        "epsilon_fim": 0.05,
        "epsilon_decay_passos": 120_000,
        "target_update": 250,
        "grad_clip": 10.0,
    },
}

# Cada variação muda somente o item indicado em relação ao perfil-base.
ALTERACOES = {
    "baseline": {},
    "exploracao_rapida": {"epsilon_decay_passos": 10_000},
    "learning_rate_alta": {"lr": 1e-2},
    "target_rapido": {"target_update": 25},
    "target_lento": {"target_update": 1_000},
}

assert AMBIENTE in PERFIS
assert set(CONFIGURACOES_ESCOLHIDAS) <= set(ALTERACOES)
assert set(ALGORITMOS) <= {"DQN", "DDQN"}

CONFIGS = {}
for nome in CONFIGURACOES_ESCOLHIDAS:
    CONFIGS[nome] = deepcopy(PERFIS[AMBIENTE])
    CONFIGS[nome].update(ALTERACOES[nome])

print("Ambiente:", AMBIENTE)
print("Configurações:", CONFIGURACOES_ESCOLHIDAS)
print("Sementes:", SEMENTES, "| Algoritmos:", ALGORITMOS)
pd.DataFrame(CONFIGS).T


## Configurações comparadas

As variações alteram uma configuração por vez: decaimento de epsilon, learning rate ou frequência de atualização da rede-alvo. O restante do protocolo é mantido.


In [ ]:
#@title 4. Sementes, replay buffer e rede neural
Transicao = namedtuple("Transicao", "estado acao recompensa proximo_estado terminal")


def fixar_semente(semente):
    random.seed(semente)
    np.random.seed(semente)
    torch.manual_seed(semente)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(semente)
    # Favorece repetibilidade; ainda pode haver pequenas diferenças entre hardware/versões.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class ReplayBuffer:
    def __init__(self, capacidade):
        self.memoria = deque(maxlen=capacidade)

    def adicionar(self, *args):
        self.memoria.append(Transicao(*args))

    def amostrar(self, tamanho):
        lote = random.sample(self.memoria, tamanho)
        estados = torch.as_tensor(np.array([x.estado for x in lote]), dtype=torch.float32, device=DISPOSITIVO)
        acoes = torch.as_tensor([x.acao for x in lote], dtype=torch.int64, device=DISPOSITIVO).unsqueeze(1)
        recompensas = torch.as_tensor([x.recompensa for x in lote], dtype=torch.float32, device=DISPOSITIVO).unsqueeze(1)
        proximos = torch.as_tensor(np.array([x.proximo_estado for x in lote]), dtype=torch.float32, device=DISPOSITIVO)
        terminais = torch.as_tensor([x.terminal for x in lote], dtype=torch.float32, device=DISPOSITIVO).unsqueeze(1)
        return estados, acoes, recompensas, proximos, terminais

    def __len__(self):
        return len(self.memoria)


class RedeQ(nn.Module):
    def __init__(self, dim_estado, n_acoes, camadas):
        super().__init__()
        modulos = []
        entrada = dim_estado
        for largura in camadas:
            modulos.extend([nn.Linear(entrada, largura), nn.ReLU()])
            entrada = largura
        modulos.append(nn.Linear(entrada, n_acoes))
        self.rede = nn.Sequential(*modulos)

    def forward(self, x):
        return self.rede(x)


In [ ]:
#@title 5. Agente DQN/DDQN (a diferença está no cálculo do alvo)
class AgenteDQN:
    def __init__(self, dim_estado, n_acoes, config, algoritmo):
        if algoritmo not in {"DQN", "DDQN"}:
            raise ValueError("algoritmo deve ser DQN ou DDQN")
        self.n_acoes = n_acoes
        self.config = config
        self.algoritmo = algoritmo
        self.online = RedeQ(dim_estado, n_acoes, config["camadas"]).to(DISPOSITIVO)
        self.alvo = RedeQ(dim_estado, n_acoes, config["camadas"]).to(DISPOSITIVO)
        self.alvo.load_state_dict(self.online.state_dict())
        self.alvo.eval()
        self.otimizador = torch.optim.Adam(self.online.parameters(), lr=config["lr"])
        self.memoria = ReplayBuffer(config["buffer"])
        self.atualizacoes = 0

    def epsilon(self, passo):
        fracao = min(passo / self.config["epsilon_decay_passos"], 1.0)
        return self.config["epsilon_inicio"] + fracao * (
            self.config["epsilon_fim"] - self.config["epsilon_inicio"]
        )

    def agir(self, estado, epsilon):
        if random.random() < epsilon:
            return random.randrange(self.n_acoes), np.nan
        with torch.no_grad():
            tensor = torch.as_tensor(estado, dtype=torch.float32, device=DISPOSITIVO).unsqueeze(0)
            qs = self.online(tensor)
        return int(qs.argmax(dim=1).item()), float(qs.max().item())

    def aprender(self):
        if len(self.memoria) < max(self.config["warmup"], self.config["batch"]):
            return None, None

        estados, acoes, recompensas, proximos, terminais = self.memoria.amostrar(self.config["batch"])
        todos_q = self.online(estados)
        q_previsto = todos_q.gather(1, acoes)
        q_max_medio = todos_q.detach().max(dim=1).values.mean()

        with torch.no_grad():
            if self.algoritmo == "DQN":
                # DQN: a rede-alvo escolhe o maior valor e o avalia.
                q_proximo = self.alvo(proximos).max(dim=1, keepdim=True).values
            else:
                # DDQN: online escolhe; alvo avalia exatamente a ação escolhida.
                acoes_online = self.online(proximos).argmax(dim=1, keepdim=True)
                q_proximo = self.alvo(proximos).gather(1, acoes_online)
            alvo_td = recompensas + self.config["gamma"] * (1.0 - terminais) * q_proximo

        perda = nn.functional.smooth_l1_loss(q_previsto, alvo_td)
        self.otimizador.zero_grad(set_to_none=True)
        perda.backward()
        nn.utils.clip_grad_norm_(self.online.parameters(), self.config["grad_clip"])
        self.otimizador.step()

        self.atualizacoes += 1
        if self.atualizacoes % self.config["target_update"] == 0:
            self.alvo.load_state_dict(self.online.state_dict())

        return float(perda.item()), float(q_max_medio.item())


In [ ]:
#@title 6. Função de treinamento e registros por episódio
def treinar(ambiente_id, algoritmo, nome_config, config, semente, mostrar_progresso=True):
    fixar_semente(semente)
    env = gym.make(ambiente_id)  # sem renderização durante o treino
    env.action_space.seed(semente)
    dim_estado = int(np.prod(env.observation_space.shape))
    agente = AgenteDQN(dim_estado, env.action_space.n, config, algoritmo)
    historico = []
    passo_global = 0

    for episodio in range(1, config["episodios"] + 1):
        inicio = time.perf_counter()
        estado, _ = env.reset(seed=semente + episodio)
        retorno = 0.0
        perdas, valores_q = [], []
        passos = 0

        while True:
            eps = agente.epsilon(passo_global)
            acao, q_max = agente.agir(estado, eps)
            proximo_estado, recompensa, terminated, truncated, _ = env.step(acao)
            acabou = terminated or truncated
            # Só terminated bloqueia bootstrap; truncated é geralmente limite de tempo.
            agente.memoria.adicionar(estado, acao, recompensa, proximo_estado, float(terminated))
            perda, q_treino = agente.aprender()
            if perda is not None:
                perdas.append(perda)
                valores_q.append(q_treino)
            elif not np.isnan(q_max):
                valores_q.append(q_max)

            estado = proximo_estado
            retorno += recompensa
            passos += 1
            passo_global += 1
            if acabou:
                break

        historico.append({
            "ambiente": ambiente_id,
            "algoritmo": algoritmo,
            "configuracao": nome_config,
            "semente": semente,
            "episodio": episodio,
            "retorno": retorno,
            "recompensa": retorno,
            "passos": passos,
            "epsilon": agente.epsilon(passo_global),
            "perda_media": float(np.mean(perdas)) if perdas else np.nan,
            "q_max_medio": float(np.mean(valores_q)) if valores_q else np.nan,
            "duracao_s": time.perf_counter() - inicio,
        })

        if mostrar_progresso and (episodio == 1 or episodio % 50 == 0):
            media = np.mean([x["retorno"] for x in historico[-20:]])
            print(f"{ambiente_id} | {nome_config} | {algoritmo} | seed={semente} | "
                  f"ep={episodio}/{config['episodios']} | média(20)={media:.1f} | eps={eps:.3f}")

    env.close()
    return agente, historico


In [ ]:
#@title 7. Smoke test muito curto (opcional)
EXECUTAR_SMOKE_TEST = False  #@param {type:"boolean"}

if EXECUTAR_SMOKE_TEST:
    config_smoke = deepcopy(PERFIS[AMBIENTE])
    config_smoke.update({
        "episodios": 3,
        "buffer": 500,
        "warmup": 32,
        "batch": 32,
        "epsilon_decay_passos": 200,
        "target_update": 10,
    })
    agente_smoke, historico_smoke = treinar(
        AMBIENTE, "DDQN", "SMOKE_NAO_VALIDO", config_smoke, 999, mostrar_progresso=False
    )
    print(pd.DataFrame(historico_smoke)[["episodio", "retorno", "perda_media"]])
    del agente_smoke, historico_smoke
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print("Smoke test não executado. Isso é normal.")


In [ ]:
#@title 8. Executar experimentos selecionados
todos_historicos = []
modelos_treinados = {}
inicio_suite = time.perf_counter()

for nome_config, config in CONFIGS.items():
    for algoritmo in ALGORITMOS:
        for semente in SEMENTES:
            print(f"\n=== {nome_config} | {algoritmo} | semente {semente} ===")
            agente, linhas = treinar(AMBIENTE, algoritmo, nome_config, config, semente)
            modelos_treinados[(AMBIENTE, nome_config, algoritmo, semente)] = agente.online.to("cpu").eval()
            todos_historicos.extend(linhas)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

df_treino = pd.DataFrame(todos_historicos)
arquivo_treino = PASTA_RESULTADOS / "treino_episodios.csv"
df_treino.to_csv(arquivo_treino, index=False)

config_exportada = {
    "ambiente": AMBIENTE,
    "configuracoes_escolhidas": CONFIGURACOES_ESCOLHIDAS,
    "sementes": SEMENTES,
    "algoritmos": ALGORITMOS,
    "episodios_avaliacao": EPISODIOS_AVALIACAO,
    "configs_efetivas": CONFIGS,
    "dispositivo_treino": str(DISPOSITIVO),
}
with open(PASTA_RESULTADOS / "configuracao.json", "w", encoding="utf-8") as f:
    json.dump(config_exportada, f, ensure_ascii=False, indent=2)

print(f"\nTreino concluído em {(time.perf_counter() - inicio_suite) / 60:.1f} min")
print("CSV salvo:", arquivo_treino)
df_treino.tail()


In [ ]:
#@title 9. Avaliação gananciosa separada (epsilon = 0)
def avaliar(modelo, ambiente_id, config, semente, n_episodios=20):
    env = gym.make(ambiente_id)
    modelo = modelo.to(DISPOSITIVO).eval()
    linhas = []

    for episodio in range(1, n_episodios + 1):
        estado, _ = env.reset(seed=100_000 + 1_000 * semente + episodio)
        with torch.no_grad():
            s0 = torch.as_tensor(estado, dtype=torch.float32, device=DISPOSITIVO).unsqueeze(0)
            q_inicial = float(modelo(s0).max().item())

        retorno = 0.0
        retorno_descontado = 0.0
        passos = 0
        sucesso = False
        while True:
            with torch.no_grad():
                s = torch.as_tensor(estado, dtype=torch.float32, device=DISPOSITIVO).unsqueeze(0)
                acao = int(modelo(s).argmax(dim=1).item())
            estado, recompensa, terminated, truncated, _ = env.step(acao)
            retorno += recompensa
            retorno_descontado += (config["gamma"] ** passos) * recompensa
            passos += 1
            acabou = terminated or truncated
            if ambiente_id == "CartPole-v1":
                sucesso = passos >= 475
            else:
                sucesso = bool(terminated and estado[0] >= 0.5)
            if acabou:
                break

        linhas.append({
            "episodio_avaliacao": episodio,
            "recompensa": retorno,
            "passos": passos,
            "sucesso": float(sucesso),
            "q_previsto_estado_inicial": q_inicial,
            "retorno_descontado_real": retorno_descontado,
            "gap_superestimacao_aprox": q_inicial - retorno_descontado,
        })

    env.close()
    modelo.to("cpu")
    return linhas


linhas_avaliacao = []
for (ambiente, nome_config, algoritmo, semente), modelo in modelos_treinados.items():
    avaliados = avaliar(modelo, ambiente, CONFIGS[nome_config], semente, EPISODIOS_AVALIACAO)
    for linha in avaliados:
        linha.update({
            "ambiente": ambiente,
            "configuracao": nome_config,
            "algoritmo": algoritmo,
            "semente": semente,
        })
        linhas_avaliacao.append(linha)

df_avaliacao = pd.DataFrame(linhas_avaliacao)
df_avaliacao.to_csv(PASTA_RESULTADOS / "avaliacao_episodios.csv", index=False)

chaves = ["ambiente", "configuracao", "algoritmo", "semente"]
df_avaliacao_resumo = (
    df_avaliacao.groupby(chaves, as_index=False)
    .agg(
        recompensa_media=("recompensa", "mean"),
        recompensa_desvio=("recompensa", "std"),
        passos_media=("passos", "mean"),
        taxa_sucesso=("sucesso", "mean"),
        q_inicial_medio=("q_previsto_estado_inicial", "mean"),
        retorno_descontado_real_medio=("retorno_descontado_real", "mean"),
        gap_superestimacao_aprox_medio=("gap_superestimacao_aprox", "mean"),
    )
)
df_avaliacao_resumo.to_csv(PASTA_RESULTADOS / "avaliacao_resumo_por_semente.csv", index=False)
df_avaliacao_resumo


### Gap Q aproximado

O gap compara `max Q(s0)` ao retorno descontado observado. É um diagnóstico descritivo sujeito à variabilidade das trajetórias, não uma prova formal de viés.


In [ ]:
#@title 10. Gráficos: média móvel e faixa entre sementes
def grafico_curvas(df, janela=20):
    arquivos = []
    for (ambiente, configuracao), grupo in df.groupby(["ambiente", "configuracao"]):
        fig, ax = plt.subplots(figsize=(10, 5.5))
        for algoritmo, dados_alg in grupo.groupby("algoritmo"):
            curvas = []
            for _, dados_seed in dados_alg.groupby("semente"):
                serie = dados_seed.sort_values("episodio").set_index("episodio")["retorno"]
                curvas.append(serie.rolling(janela, min_periods=1).mean().rename(str(len(curvas))))
            matriz = pd.concat(curvas, axis=1)
            media = matriz.mean(axis=1)
            ax.plot(media.index, media, linewidth=2, label=algoritmo)
            if matriz.shape[1] > 1:
                ax.fill_between(media.index, matriz.min(axis=1), matriz.max(axis=1), alpha=0.18)

        ax.set(title=f"{ambiente} — {configuracao}", xlabel="Episódio", ylabel=f"Retorno (média móvel {janela})")
        ax.grid(alpha=0.25)
        ax.legend(title="Algoritmo")
        fig.tight_layout()
        nome = PASTA_RESULTADOS / f"curva_{ambiente}_{configuracao}.png"
        fig.savefig(nome, dpi=160, bbox_inches="tight")
        plt.show()
        plt.close(fig)
        arquivos.append(nome)
    return arquivos


arquivos_curvas = grafico_curvas(df_treino)

fig, ax = plt.subplots(figsize=(9, 5))
dados_barra = (
    df_avaliacao.groupby(["configuracao", "algoritmo"])["recompensa"]
    .agg(["mean", "std"])
    .reset_index()
)
pivo = dados_barra.pivot(index="configuracao", columns="algoritmo", values="mean")
pivo.plot(kind="bar", ax=ax, rot=20)
ax.set(title=f"Avaliação gananciosa — {AMBIENTE}", xlabel="Configuração", ylabel="Recompensa média")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
arquivo_barra = PASTA_RESULTADOS / "avaliacao_recompensa_media.png"
fig.savefig(arquivo_barra, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print("PNGs salvos:", len(arquivos_curvas) + 1)


In [ ]:
#@title 11. Tabela agregada final
resumo_final = (
    df_avaliacao.groupby(["ambiente", "configuracao", "algoritmo"], as_index=False)
    .agg(
        sementes=("semente", "nunique"),
        recompensa_media=("recompensa", "mean"),
        recompensa_desvio=("recompensa", "std"),
        passos_media=("passos", "mean"),
        taxa_sucesso=("sucesso", "mean"),
        q_inicial_medio=("q_previsto_estado_inicial", "mean"),
        retorno_descontado_real_medio=("retorno_descontado_real", "mean"),
        gap_superestimacao_aprox_medio=("gap_superestimacao_aprox", "mean"),
    )
    .sort_values(["configuracao", "algoritmo"])
)
resumo_final.to_csv(PASTA_RESULTADOS / "resumo_final.csv", index=False)
display(resumo_final.round(3))



In [ ]:
#@title 12. Compactar todos os artefatos e baixar no Colab
identificador = f"{AMBIENTE}_{'_'.join(CONFIGURACOES_ESCOLHIDAS)}".replace("-", "_")
arquivo_zip = Path(f"/content/resultados_dqn_ddqn_{identificador}.zip")
if arquivo_zip.exists():
    arquivo_zip.unlink()
shutil.make_archive(str(arquivo_zip.with_suffix("")), "zip", root_dir=PASTA_RESULTADOS)
print("ZIP criado:", arquivo_zip, f"({arquivo_zip.stat().st_size / 1e6:.2f} MB)")

try:
    from google.colab import files
    files.download(str(arquivo_zip))
except ImportError:
    print("Fora do Colab: o download automático não foi iniciado.")
